# 🧠 Semantic Entropy Probes — Live Demo

This notebook runs the **full Semantic Probes app** on Google Colab's free GPU and gives you a **public URL** to access it from any device.

### Prerequisites
1. **GPU Runtime**: Go to `Runtime → Change runtime type → T4 GPU`
2. **HuggingFace Token**: You'll need your HF token to download Llama 3.1
3. **ngrok Token**: Free signup at [ngrok.com](https://ngrok.com) → get your authtoken
4. **Probe Files**: Upload your `.pkl` probe files when prompted

### How to Use
Just run all cells in order (`Runtime → Run all`), or run them one by one. The final cell will print a **public URL** you can open on any browser.

---
## Step 1: Verify GPU & Clone Repository

In [ ]:
# Verify GPU is available
import torch
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    gpu_mem = torch.cuda.get_device_properties(0).total_mem / 1e9
    print(f"✅ GPU: {gpu_name} ({gpu_mem:.1f} GB)")
else:
    print("❌ No GPU detected! Go to Runtime → Change runtime type → T4 GPU")
    raise RuntimeError("GPU required")

In [ ]:
# Clone the repository
import os

REPO_DIR = "/content/semantic-entropy-probes"

if os.path.exists(REPO_DIR):
    print("Repository already cloned. Pulling latest...")
    !cd {REPO_DIR} && git pull
else:
    !git clone https://github.com/DangerDudeSL/semantic-entropy-probes.git {REPO_DIR}
    print("✅ Repository cloned")

os.chdir(REPO_DIR)
print(f"Working directory: {os.getcwd()}")

---
## Step 2: Install Dependencies

In [ ]:
# Install Python dependencies
# Colab already has torch + CUDA, so we skip those
!pip install -q transformers accelerate bitsandbytes fastapi uvicorn pydantic scikit-learn nltk psutil sentencepiece protobuf requests pyngrok aiofiles

# Install spaCy for sentence segmentation
!pip install -q spacy
!python -m spacy download en_core_web_sm 2>/dev/null

# Install pysbd as fallback segmenter
!pip install -q pysbd

print("✅ Python dependencies installed")

In [ ]:
# Install Node.js and build tools for the frontend
!apt-get update -qq && apt-get install -y -qq nodejs npm > /dev/null 2>&1

# Install a newer Node.js version (Colab's default may be old)
!curl -fsSL https://deb.nodesource.com/setup_20.x | bash - > /dev/null 2>&1
!apt-get install -y -qq nodejs > /dev/null 2>&1

!node --version
!npm --version
print("✅ Node.js installed")

---
## Step 3: Upload Probe Files (.pkl)

The probe files are not in the GitHub repo (they're too large). You have two options:

**Option A (Recommended):** Upload from Google Drive  
**Option B:** Upload directly from your computer

In [ ]:
# ===== OPTION A: Upload from Google Drive =====
# First, upload your .pkl files to your Google Drive,
# then uncomment and run this cell.

# from google.colab import drive
# drive.mount('/content/drive')

# # Copy probe files from Drive to the correct location
# import shutil
# PROBE_DIR = f"{REPO_DIR}/semantic_entropy_probes/models"
# os.makedirs(PROBE_DIR, exist_ok=True)

# # UPDATE THESE PATHS to match where you put the files in Google Drive
# shutil.copy('/content/drive/MyDrive/probes/Llama3-8b_inference.pkl', PROBE_DIR)
# shutil.copy('/content/drive/MyDrive/probes/Llama2-7b_inference.pkl', PROBE_DIR)
# print("✅ Probe files copied from Google Drive")


# ===== OPTION B: Upload directly from your computer =====
# Uncomment and run this to upload via browser file picker

from google.colab import files
import shutil

PROBE_DIR = f"{REPO_DIR}/semantic_entropy_probes/models"
os.makedirs(PROBE_DIR, exist_ok=True)

print("📁 Please upload your probe .pkl files (Llama3-8b_inference.pkl and/or Llama2-7b_inference.pkl)")
uploaded = files.upload()

for filename in uploaded.keys():
    dest = os.path.join(PROBE_DIR, filename)
    shutil.move(filename, dest)
    print(f"  ✅ {filename} → {dest}")

print(f"\n📋 Files in {PROBE_DIR}:")
for f in os.listdir(PROBE_DIR):
    print(f"  - {f}")

---
## Step 4: HuggingFace Login

Required to download Llama 3.1. Make sure you've accepted the [Llama 3.1 license](https://huggingface.co/meta-llama/Meta-Llama-3.1-8B-Instruct) on HuggingFace.

In [ ]:
# Login to HuggingFace
from huggingface_hub import login

# Enter your HuggingFace token when prompted
login()

---
## Step 5: Build Frontend

In [ ]:
# Build the React frontend
os.chdir(f"{REPO_DIR}/frontend")

!npm install 2>&1 | tail -5
!npm run build 2>&1 | tail -10

# Verify build output
dist_path = f"{REPO_DIR}/frontend/dist"
if os.path.exists(dist_path):
    print(f"\n✅ Frontend built successfully!")
    !ls -la {dist_path}
else:
    print("❌ Frontend build failed!")

os.chdir(REPO_DIR)

---
## Step 6: Setup ngrok Tunnel

ngrok creates a public URL that tunnels to your Colab instance.

Get your **free authtoken** from: https://dashboard.ngrok.com/get-started/your-authtoken

In [ ]:
# Configure ngrok
from pyngrok import ngrok
import getpass

# Enter your ngrok auth token
NGROK_TOKEN = getpass.getpass("Enter your ngrok authtoken: ")
ngrok.set_auth_token(NGROK_TOKEN)
print("✅ ngrok configured")

---
## Step 7: 🚀 Launch the App!

This cell starts the FastAPI server (which serves both the API and the React frontend) and creates a public URL.

**⏱ First launch takes ~3-5 minutes** to download and load the Llama 3.1 model.

Once you see the public URL, **click it to open the app!**

In [ ]:
import subprocess
import time
import requests
from pyngrok import ngrok

os.chdir(REPO_DIR)

# Start the FastAPI backend (which also serves the frontend)
print("🔄 Starting the server...")
print("   This will download & load Llama 3.1 8B (~5 GB) + DeBERTa NLI model.")
print("   First run takes 3-5 minutes. Subsequent runs are faster (cached).\n")

server_process = subprocess.Popen(
    ["python", "backend/main.py"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1
)

# Wait for server to be ready (check /status endpoint)
print("⏳ Waiting for model to load...")
max_wait = 600  # 10 minutes max
start_time = time.time()
ready = False

while time.time() - start_time < max_wait:
    try:
        response = requests.get("http://localhost:8000/status", timeout=3)
        if response.status_code == 200:
            data = response.json()
            if data.get("model_loaded"):
                ready = True
                break
            else:
                elapsed = int(time.time() - start_time)
                print(f"   [{elapsed}s] Server is up, model still loading...")
    except:
        elapsed = int(time.time() - start_time)
        if elapsed % 30 == 0:
            print(f"   [{elapsed}s] Server starting...")
    time.sleep(5)

if not ready:
    print("❌ Server didn't become ready in time. Check logs above.")
else:
    elapsed = int(time.time() - start_time)
    print(f"\n✅ Model loaded in {elapsed} seconds!")
    
    # Create ngrok tunnel
    public_url = ngrok.connect(8000)
    
    print("\n" + "="*60)
    print("  🌐 YOUR APP IS LIVE!")
    print(f"  📱 Public URL: {public_url}")
    print("="*60)
    print("\n  Open this URL on any device (phone, laptop, etc.)")
    print("  The app will stay live as long as this notebook is running.")
    print("\n  To stop: Runtime → Interrupt execution")

---
## 📊 Monitor Server Logs

Run this cell to see what the server is doing (useful for debugging).

In [ ]:
# Print recent server output (run this cell anytime to check logs)
import subprocess
print("Recent server output:")
print("-" * 40)
try:
    # Read any available output from server
    import select
    while True:
        line = server_process.stdout.readline()
        if line:
            print(line.strip())
        else:
            break
except:
    print("Server process not accessible. It may have stopped.")

---
## 🛑 Stop the App
Run this cell to cleanly shut down everything.

In [ ]:
# Clean shutdown
ngrok.kill()
server_process.terminate()
print("✅ Server and tunnel stopped.")